In [7]:
import pandas as pd
import re
import shlex
import csv

csv_path = '2022/Chs2022ecl_pumf.csv'
label_path = '2022/Layouts/STATA/chs2022ecl_pumf_lbe.do'
val_path = '2022/Layouts/STATA/chs2022ecl_pumf_vale.do'
df = pd.read_csv(csv_path)

In [8]:
# Map coded CSV values to answer categories using STATA files

# Parse value labels
with open(val_path, 'r', encoding='latin-1') as f:
    content = f.read().replace('///', '')

label_pool = {}
def_blocks = re.findall(r'label define\s+(\w+)\s+(.*?)(?=label|$)', content, re.DOTALL | re.IGNORECASE)
for lbl_name, mapping_str in def_blocks:
    pairs = re.findall(r'(\d+(?:\.\d+)?)\s+"(.*?)"', mapping_str)
    if pairs:
        label_pool[lbl_name.upper()] = {float(val): text for val, text in pairs}

var_links = {}
link_matches = re.findall(r'label values (\w+) (\w+)', content, re.IGNORECASE)
for var, lbl in link_matches:
    var_links[var.upper()] = lbl.upper()


# Apply mapping to values
mapped_df = df.copy()
mapped_columns = []
unmapped_columns = []

for col in mapped_df.columns:
    col_upper = col.upper()
    target_label = var_links.get(col_upper)
    
    if not target_label and f"{col_upper}_FMT" in label_pool:
        target_label = f"{col_upper}_FMT"
        
    if target_label and target_label in label_pool:
        mapping_dict = label_pool[target_label]
        numeric_series = pd.to_numeric(mapped_df[col], errors='coerce')
        mapped_df[col] = numeric_series.map(mapping_dict).fillna(mapped_df[col])
        mapped_columns.append(col)
    else:
        unmapped_columns.append(col)

# Write mapped values to CSV
mapped_df.head()

,PUMFID,EHA_10,EHA_25,FP_05,DWI_05A,DWI_05B,DWI_05C,DWI_05D,NEI_05A,NEI_05B,...,PSTIR_GR,PVISMIN,PWSA_D15,P2DCT_20,P2DCT_25,PATT_05,PATT_10,PATT_15A,PATT_15B,VERDATE
0,63501,Easy,No,Yes,No,No,No,No,Not a problem,Not a problem,...,Spending less than 30% of income on shelters c...,Not stated,999.6,Valid skip,Valid skip,One vehicle,All the time,Valid skip,Valid skip,11/08/2025
1,63502,Neither difficult nor easy,No,No,No,No,No,No,Not a problem,Not a problem,...,Spending less than 30% of income on shelters c...,No household member belongs to a visible minor...,999.6,Valid skip,Valid skip,Two or more vehicles,All the time,Valid skip,Valid skip,11/08/2025
2,63503,Neither difficult nor easy,No,Yes,No,No,No,No,Not a problem,Not a problem,...,Spending less than 30% of income on shelters c...,No household member belongs to a visible minor...,999.6,Valid skip,Valid skip,Two or more vehicles,All the time,Valid skip,Valid skip,11/08/2025
3,63504,Neither difficult nor easy,No,Yes,No,Yes,No,No,A small problem,Not a problem,...,Spending less than 30% of income on shelters c...,At least 1 hhld member belongs to a visible mi...,999.6,Valid skip,Valid skip,Two or more vehicles,All the time,Valid skip,Valid skip,11/08/2025
4,63505,Easy,No,Yes,No,No,No,No,Not a problem,Not a problem,...,Spending less than 30% of income on shelters c...,No household member belongs to a visible minor...,999.6,Two,"No, only regular maintenance is needed...",One vehicle,Sometimes,Valid skip,A little,11/08/2025


In [9]:
# Parse variable labels
rows = []
variable_description_map = {}

with open(label_path, "r", encoding="latin-1") as f:
    var_labels_content = f.readlines()

for line in var_labels_content:
    temp = line.strip()
    if not temp or not temp.lower().startswith("label variable"):
        continue
    try:
        parts = shlex.split(temp)
        if len(parts) >= 4:
            variable = parts[2].upper()
            description = parts[3]
            rows.append([variable, description])
            variable_description_map[variable] = description
    except ValueError:
        continue

# Write variables and descriptions to CSV
with open("chs2022_labels.csv", "w", newline="", encoding="latin-1") as f:
    writer = csv.writer(f)
    writer.writerow(["variable", "description"])
    writer.writerows(rows)

# Check for unmapped columns
for col in unmapped_columns:
    desc = variable_description_map.get(col.upper())
    print(f"Variable: {col:<12} | Description: {desc}")

Variable: PUMFID       | Description: Unique household identifier
Variable: REGION       | Description: Region of residence
Variable: PDV_SHCO     | Description: Shelter costs - $ monthly amount (all costs included)
Variable: PDWLTYPE     | Description: Dwelling type
Variable: PFWEIGHT     | Description: Household weight
Variable: PHTYPE       | Description: Household type
Variable: PPROV        | Description: Province of residence
Variable: PSCR_D40     | Description: Months spent on waitlist before obtaining subsidized housing
Variable: PWSA_D15     | Description: Waitlist for SAH - months currently on waitlist
Variable: VERDATE      | Description: Version date


In [10]:
mapped_df_fin = mapped_df.copy()

mapping_dict_fin = {
    "REGION": {
        1: "Atlantic",
        2: "Quebec",
        3: "Ontario",
        4: "Praries",
        5: "British Columbia"
    },
    "PDV_SHCO": {
        9999999.99: "Not stated" 
    },
    "PDWLTYPE": {
        1: "Single-detached house",
        2: "Semi-detached house or other single-attached house",
        3: "Row house",
        4: "Apartment or flat in a duplex",
        5: "Apartment in a building that has five or more storeys",
        6: "Aparment in a building that has fewer than five storeys",
        99: "Not stated"
    },
    "PHHSIZE": {
        99: "Not stated"
    },
    "PHTYPE": {
        1: "One couple (married or common-law) with children",
        2: "One couple (married or common-law) without children",
        3: "One lone-parent family",
        4: "One census family plus additional persons",
        5: "One-person (not in census family)",
        6: "Two or more persons not in census families",
        99: "Not stated"
    },
    "PPROV": {
        10: "Newfoundland and Labrodor",
        11: "Prince Edward Island",
        12: "Nova Scotia",
        13: "New Brunswick",
        24: "Quebec",
        35: "Ontario",
        46: "Manitoba",
        47: "Saskatchewan",
        48: "Alberta",
        59: "British Columbia"
    },
    "PSCR_D40": {
        999.6: "Valid skip",
        999.9: "Not stated"
    },
    "PWSA_D15": {
        999.6: "Valid skip",
        999.9: "Not stated"
    }
}

val_count = 0

# Finish mapping codes to values
for col, mapping in mapping_dict_fin.items():
    if col in mapped_df.columns:
        numeric_series = pd.to_numeric(mapped_df_fin[col], errors="coerce")
        mapped_df_fin[col] = numeric_series.map(mapping).fillna(mapped_df_fin[col])

# Replace skips and not-stated values with NA
mapped_df_fin = mapped_df_fin.replace({
    # "Valid skip": pd.NA,
    "Not stated": pd.NA
})

mapped_df_fin.to_csv('Chs2022ecl_pumf_VALUES_MAPPED_FINAL.csv', index=False, encoding='utf-8-sig')